# PiPER-X (`pyAgxArm` V189) 全功能交互式测试 Notebook

覆盖 `pyAgxArm`（PiPER-X / `PiperFW.V189`）主臂 `Driver` 与 `agx_gripper` 夹爪 `Driver` 的全部功能接口。

## 0. 激活 1 号机械臂（连接 `can0` + 恢复 Follower 反馈推送 + 初始化夹爪 + 使能）

In [ ]:
import time
from typing import Any
from pyAgxArm import AgxArmFactory, ArmModel, PiperFW, create_agx_arm_config

config = create_agx_arm_config(
    robot=ArmModel.PIPER_X,
    firmeware_version=PiperFW.V189,
    interface='socketcan',
    channel='can0',
    auto_connect=False,
)
robot: Any = AgxArmFactory.create_arm(config)
robot.connect()

# 若机械臂上次停留在 Leader(0xFA) 模式，固件会关闭周期状态反馈推送（FPS=0）；
# 先发 set_follower_mode()(0xFC) 可立即恢复 200Hz 状态反馈推送。
# move_j / move_p / move_l / move_c / move_mit 全是 Follower 模式下的
robot.set_follower_mode()

# FPS = 每秒收到的 CAN 反馈帧数（正常约 2100 Hz）；get_firmware() 内部要求 FPS > 0 才会解析返回
while robot.get_fps() <= 0:
    time.sleep(0.02)

# init_effector 同一实例只能调用一次，重复调用会抛 RuntimeError
gripper: Any = robot.init_effector('agx_gripper')

# 255(0xFF) 为 SDK 约定的“全部 6 个关节”特殊枚举值（非位掩码，底层发 0x471.B0=7）；单轴可传 1..6
robot.enable(255)
time.sleep(0.2)

print("Connected:", robot.is_connected(), "| OK:", robot.is_ok(), "| FPS:", robot.get_fps())
print("Firmware :", robot.get_firmware())
print("Joints enable:", robot.get_joints_enable_status_list())

---
## 1. 基础信息与实时状态反馈（只读）

In [ ]:
# 1.1 通信状态、配置与枚举选项
# 注：robot.get_driver_version() 在 SDK 抽象基类中未实现，直接调用会抛 NotImplementedError
print("type          :", robot.get_type())
print("channel       :", robot.get_channel())
print("joint_nums    :", robot.joint_nums)
print("has_comm_error:", robot.has_comm_error(), "| comm_error:", robot.get_comm_error())
print("MOTION_MODE   :", robot.OPTIONS.MOTION_MODE.value_list())
print("INSTALL_POS   :", robot.OPTIONS.INSTALLATION_POS.value_list())
print("PAYLOAD       :", robot.OPTIONS.PAYLOAD.value_list())
print("config        :", robot.get_config())

In [ ]:
# 1.2 整臂状态反馈 (RX 0x2A1: get_arm_status)
# 注：返回内存缓存的最新一帧 0x2A1（若在 Leader 模式下停止推送，读到的是切换前的旧缓存）；
#     0x2A1 不含电机使能位，电机是否使能需看 1.5 的 0x261~0x266 驱动器状态。
status = robot.get_arm_status()
print(status.msg if status else None)

In [ ]:
# 1.3 六关节实测角 (RX 0x2A5~0x2A7) 与 IK 关节角反馈 (RX 0x2AA~0x2AC)
# 注：get_ik_joint_angles() 仅在固件执行笛卡尔逆解并推送 0x2AA~0x2AC 时有值，否则为 None
joints = robot.get_joint_angles()
ik_joints = robot.get_ik_joint_angles()
print("实测关节角 (rad):", [round(x, 4) for x in joints.msg] if joints else None)
print("IK 关节角  (rad):", [round(x, 4) for x in ik_joints.msg] if ik_joints else None)

In [ ]:
# 1.4 法兰位姿 (RX 0x2A2~0x2A4)、正运动学 fk() 与 TCP 坐标变换
# 注：fk()、set_tcp_offset()、get_tcp_pose()、get_flange2tcp_pose()、get_tcp2flange_pose()
#     均为主机侧纯数学计算，不发送任何 CAN 报文，也不会自动改变 move_p/move_l 的法兰目标。
flange = robot.get_flange_pose()
joints = robot.get_joint_angles()
print("实测法兰位姿:", [round(x, 4) for x in flange.msg] if flange else None)
print("FK 法兰位姿 :", [round(x, 4) for x in robot.fk(list(joints.msg))] if joints else None)

robot.set_tcp_offset([0.0, 0.0, 0.10, 0.0, 0.0, 0.0])
print("TCP 位姿 (+0.1m Z):", [round(x, 4) for x in robot.get_tcp_pose().msg])
t_pose = robot.get_flange2tcp_pose(list(flange.msg))
print("tcp -> flange 还原:", [round(x, 4) for x in robot.get_tcp2flange_pose(t_pose)])
robot.set_tcp_offset([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

In [ ]:
# 1.5 高速电机状态 (RX 0x251~0x256) 与低速驱动器状态 (RX 0x261~0x266)
# 注：m.msg.torque 由电流乘以系数 k*b*c 估算（含重力与摩擦，非纯外力矩）；
#     关节使能状态来自 0x261~0x266 的 d.msg.foc_status.driver_enable_status。
for i in range(1, 7):
    m = robot.get_motor_states(i)
    d = robot.get_driver_states(i)
    print(f"J{i}: enabled={robot.get_joint_enable_status(i)} | "
          f"vel={m.msg.velocity:.3f} rad/s, cur={m.msg.current:.3f} A, torque={m.msg.torque:.3f} Nm | "
          f"vol={d.msg.vol:.1f} V, temp={d.msg.motor_temp}°C")

---
## 2. 运行时开关与电机使能/失能 (`0x471`)

In [ ]:
# 2.1 自动切模式开关 & 软件关节限位开关
# 注：auto_set_motion_mode_enabled 默认为 True，开启时调用任何 move_* 都会在内部自动先调对应的 set_motion_mode()；
#     joint_limits_enabled 默认为 False（关闭时普通关节目标仍按 ±2π 裁剪，开启后按 config['joint_limits'] 裁剪）。
robot.set_auto_set_motion_mode_enabled(True)
robot.set_joint_limits_enabled(True)
print("auto_set_motion_mode:", robot.get_auto_set_motion_mode_enabled())
print("joint_limits_enabled:", robot.get_joint_limits_enabled())

In [ ]:
# 2.2 关节使能与失能 (enable / disable)
# 注：仅支持传入离散值 1..6（单关节）或 255（全部 6 个关节，非位掩码）；
#     0x471 只控制 6 个关节电机，不控制夹爪（夹爪使能由 5.2/5.3 的 0x159 控制）。
robot.enable(255)
# robot.disable(6)
print("Joints enable:", robot.get_joints_enable_status_list())

---
## 3. 模式控制、速度百分比与安装方向 (`0x151`)

In [ ]:
# 3.1 设置运动速度百分比 (0x151.B2: move_spd_rate_ctrl，范围 0..100，SDK 初值 50)
# 注：仅对 p / j / l / c 四种位置速度规划模式生效（对 mit / js / cpv 不生效）；
#     调用时临时以 B1(move_mode)=0xFF 发送 0x151（只更新速度百分比、不切换运动模式），并缓存在 SDK 中供后续复用。
robot.set_speed_percent(20)

In [ ]:
# 3.2 设置安装方向 (0x151.B5: 'horizontal' | 'left' | 'right')
# 注：同样临时以 B1(move_mode)=0xFF 发送 0x151，发送后 SDK 缓存中的 installation_pos 自动复位为 0。
robot.set_installation_pos(robot.OPTIONS.INSTALLATION_POS.HORIZONTAL)

In [ ]:
# 3.3 显式切换运动模式 (0x151.B1/B3: 'p' | 'j' | 'l' | 'c' | 'mit' | 'js' | 'cpv')
# 注：1) 当 auto_set_motion_mode_enabled=True 时，直接调第 4 节的 move_* 会自动调用本函数，无需手动先调；
#     2) SDK 对相同载荷的 0x151 帧有 0.1s 最小重发间隔过滤；
#     3) 0x2A1 只反馈 B1(mode_feedback)，不反馈 B3(mit_mode)，因此 'j' 与 'js' 的 mode_feedback 均显示为 MOVE_J(0x1)。
robot.set_motion_mode(robot.OPTIONS.MOTION_MODE.J)
time.sleep(0.1)
print(robot.get_arm_status().msg)

---
## 4. 七种 `move_mode` 运动控制

In [ ]:
# 4.1 move_j (move_mode=1, mit_mode=0x00, TX 0x155~0x157)：关节空间运动
# 注：驱动器走位置-速度-电流串级 PI 闭环（含积分项自动消除稳态误差，高刚度锁位）；运动速度由 set_speed_percent 控制。
q = list(robot.get_joint_angles().msg)
q[5] += 0.05  # J6 +0.05 rad
robot.move_j(q)

"""
这个wrap内部进行了模式切换
def move_p(self, pose: List[float]):
    msgs = self._deal_move_p_msgs(pose)
    self._maybe_set_motion_mode('p')  # 内部自动执行 self.set_motion_mode('p')
    self._send_msgs(msgs)
"""

In [ ]:
# 4.2 move_js (move_mode=1, mit_mode=0xAD, TX 0x155~0x157)：关节快速响应模式
# 注：与 move_j 共用 0x155~0x157 目标帧，但开启 mit_mode=0xAD，无平滑与轨迹规划，直接快速响应，单次目标跳变不宜过大。
q = list(robot.get_joint_angles().msg)
q[5] -= 0.05  # J6 -0.05 rad
robot.move_js(q)

In [ ]:
# 4.3 move_p (move_mode=0, mit_mode=0x00, TX 0x152~0x154)：笛卡尔点到点运动
# 注：输入基座到法兰位姿 [x, y, z, roll, pitch, yaw]（roll/yaw ∈ [-π, π], pitch ∈ [-π/2, π/2]），
#     由固件逆解并插补到目标位姿，不保证末端轨迹走直线；速度由 set_speed_percent 控制。
pose = list(robot.get_flange_pose().msg)
pose[2] += 0.01  # Z +1 cm
robot.move_p(pose)

In [ ]:
# 4.4 move_l (move_mode=2, mit_mode=0x00, TX 0x152~0x154)：笛卡尔直线运动
# 注：与 move_p 发送完全相同的 0x152~0x154 位姿帧，由 0x151.B1=2 指定末端沿直线插补到达目标。
pose = list(robot.get_flange_pose().msg)
pose[2] -= 0.01  # Z -1 cm
robot.move_l(pose)

In [ ]:
# 4.5 move_c (move_mode=3, mit_mode=0x00, TX 0x152~0x154 + 0x158)：笛卡尔圆弧运动
# 注：内部依次发送 起点(0x158.B0=1) -> 中点(0x158.B0=2) -> 终点(0x158.B0=3) 三组法兰位姿确定圆弧轨迹。
sp = list(robot.get_flange_pose().msg)
mp = [sp[0], sp[1] + 0.01, sp[2] + 0.01, sp[3], sp[4], sp[5]]
ep = [sp[0], sp[1],        sp[2] + 0.02, sp[3], sp[4], sp[5]]
robot.move_c(sp, mp, ep)

In [ ]:
# 4.6 move_mit (move_mode=6, mit_mode=0xAD, TX 0x15A~0x15F)：单关节 MIT 阻抗/力矩控制
# 注：无积分项的 PD + 前馈控制 T_ref = kp*(p_des - p) + kd*(v_des - v) + t_ff；
#     t_ff 直接为减速后关节输出端力矩(Nm)，而 SDK 原始 kp/kd 作用于减速前电机侧尺度，
#     经减速器输出到关节端时实际刚度/阻尼会放大 [22.33, 13.05, 25.47, 3.03, 1.0, 1.0] 倍。
q6 = robot.get_joint_angles().msg[5]
robot.move_mit(joint_index=6, p_des=q6, v_des=0.0, kp=0.7, kd=0.2, t_ff=0.0)

In [ ]:
# 4.7 CPV 模式 (move_mode=5, mit_mode=0x00, TX/RX 0x181~0x186)：单关节参数读写与位置/速度控制
# 注：1) get_cpv_* 默认有 min_interval=1.0s 请求节流，连续查询需传 min_interval=0.0；
#     2) move_cpv_vel(j_idx, vel) 会让关节按指定速度持续运动，停止需发 move_cpv_vel(j_idx, 0.0)。
j_idx = 6
cpv_acc = robot.get_cpv_acc(j_idx, min_interval=0.0)
print("J6 CPV:", {
    "po": robot.get_cpv_pos(j_idx, min_interval=0.0),
    "sp": robot.get_cpv_vel(j_idx, min_interval=0.0),
    "ac": cpv_acc,
    "dc": robot.get_cpv_dcc(j_idx, min_interval=0.0),
    "vv": robot.get_cpv_cv(j_idx, min_interval=0.0),
    "pp": robot.get_cpv_pp(j_idx, min_interval=0.0),
    "kp": robot.get_cpv_kp(j_idx, min_interval=0.0),
    "ki": robot.get_cpv_ki(j_idx, min_interval=0.0),
})
if cpv_acc is not None:
    print("set_cpv_acc:", robot.set_cpv_acc(j_idx, cpv_acc))

q6 = robot.get_joint_angles().msg[5]
robot.move_cpv_pos(j_idx, q6)
robot.move_cpv_vel(j_idx, 0.0)

---
## 5. 夹爪 (`agx_gripper`) 功能 (`0x159` / `0x2A8` / `0x47D` / `0x47E`)

In [ ]:
# 5.1 读取夹爪状态反馈 (0x2A8) 与控制帧回显 (0x159)
# 注：get_gripper_ctrl_states() 仅在收到外部发到总线上的 0x159 控制帧时才有值（本机发送默认不回环，常为 None）。
print("status:", gripper.get_gripper_status().msg)
print("ctrl  :", gripper.get_gripper_ctrl_states())

In [ ]:
# 5.2 夹爪开度控制 (move_gripper_m 宽度模式 / move_gripper_deg 角度模式)
# 注：夹爪控制独立于主臂 move_mode；调用 move_gripper_m 会自带使能标志 (0x159.B6=1) 驱动夹爪。
w = float(gripper.get_gripper_status().msg.value)
gripper.move_gripper_m(value=min(0.08, w + 0.005), force=0.5)
# gripper.move_gripper_deg(value=10.0, force=0.5)

In [ ]:
# 5.3 夹爪失能 (disable_gripper) 与清错重置 (reset_gripper)
# 注：disable_gripper() 发送 0x159.B6=0 撤销夹爪使能；后续再次调用 move_gripper_m() 会自动重新使能夹爪。
print("disable_gripper:", gripper.disable_gripper())
# print("reset_gripper  :", gripper.reset_gripper())

In [ ]:
# 5.4 夹爪/示教器参数读写 (0x47D / 0x47E)
# 注：set_gripper_teaching_pendant_param 要求 teaching_range_per ∈ [100, 200]、
#     max_range_config ∈ {0, 0.07, 0.1} (单位 m)、teaching_friction ∈ [1..10]。
tp = gripper.get_gripper_teaching_pendant_param(min_interval=0.0)
print("teaching_pendant_param:", tp.msg if tp else None)
if tp is not None and float(tp.msg.max_range_config) in (0, 0.07, 0.1):
    print("set_param:", gripper.set_gripper_teaching_pendant_param(
        teaching_range_per=int(tp.msg.teaching_range_per),
        max_range_config=float(tp.msg.max_range_config),
        teaching_friction=int(tp.msg.teaching_friction),
    ))

In [ ]:
# 5.5 夹爪零点标定 (calibrate_gripper, 0x159.B7=0xAE)
# ⚠️ 警告：会将夹爪当前物理位置设为 0 位！必须先手动或发指令将夹爪完全闭合后再取消注释执行：
# print("calibrate_gripper:", gripper.calibrate_gripper())

---
## 6. 固件原生零力拖动 (`Leader/Follower`)、一键回零、急停与示教记录 (`0x470` / `0x191` / `0x150`)

In [ ]:
# 6.1 进入固件原生零力拖动模式 (0x470 B0=0xFA: set_leader_mode + 0x191 mode=0: restore_leader_drag_mode)
# 注：1) 必须同时配合 set_leader_mode() 与 restore_leader_drag_mode()：若之前执行过 move_leader_to_home() 未恢复，
#        单调 set_leader_mode() 仍会保持在回零位置锁定状态，不会释放为零力拖动；
#     2) 进入 Leader 模式后，固件停止推送 0x25x/0x26x/0x2Ax 状态反馈帧（get_fps() 降为 0，get_arm_status() 仅返回旧缓存），
#        改为向外发送 0x155~0x157 关节角目标帧（可通过 get_leader_joint_angles() 读取）。
robot.set_leader_mode()
time.sleep(0.1)
robot.restore_leader_drag_mode()
time.sleep(0.2)
print("Leader 输出关节角 (RX 0x155~0x157):", robot.get_leader_joint_angles())

In [ ]:
# 6.2 固件原生主臂一键回零 (0x191 mode=1: move_leader_to_home)
# 注：1) move_leader_to_home() (0x191) 只在机械臂处于 Leader 模式 (0x470: 0xFA) 下才生效（Follower 模式下发送会被忽略）；
#     2) 触发后固件从零力拖动切入内部位置闭环自动回零位 [0, 0, 0, 0, 0, 0]，到位后继续保持位置闭环锁死（手推不动），
#        需调用下一格 6.3 的 restore_leader_drag_mode() 才会重新释放为零力拖动。
robot.set_leader_mode()
time.sleep(0.1)
robot.move_leader_to_home()  # 主从双臂同时回零可用 robot.move_leader_follower_to_home()

In [ ]:
# 6.3 从回零锁定状态恢复主臂零力拖动 (0x191 mode=0: restore_leader_drag_mode)
# 注：仅在 Leader 模式下执行过 move_leader_to_home() 锁死后使用，将主臂重新释放为可手推的零力拖动状态。
robot.restore_leader_drag_mode()

In [ ]:
# 6.4 退出 Leader 模式，切回 Follower 正常受控模式 (0x470 B0=0xFC: set_follower_mode)
# 注：退出零力拖动并保持当前姿态，同时立即恢复 0x25x/0x26x/0x2Ax 周期状态反馈推送（FPS 恢复约 2100 Hz）；
#     切换过程不会掉电机使能，无需重新调用 enable() 或重启电源。
robot.set_follower_mode()
time.sleep(0.1)
print(robot.get_arm_status().msg)

In [ ]:
# 6.5 电子快速急停 (0x150.B0=1: electronic_emergency_stop) 与复位恢复 (0x150.B0=2: reset)
# 注：electronic_emergency_stop() 触发阻尼急停（arm_status 变为 EMERGENCY_STOP(0x1)）；
#     reset() 清除急停状态，但同时会使所有关节电机失能（掉电），恢复控制前必须重新调用 robot.enable(255)。
robot.electronic_emergency_stop()
time.sleep(0.3)
print("急停后 arm_status:", robot.get_arm_status().msg.arm_status)

robot.reset()
time.sleep(0.3)
robot.enable(255)
print("复位并重新使能后 arm_status:", robot.get_arm_status().msg.arm_status)

In [ ]:
# 6.6 示教记录状态标志控制 (0x150.B2: ArmMsgMotionCtrl.grag_teach_ctrl)
# 注：在 CAN_CTRL 或待机下发送 grag_teach_ctrl=1/2 仅修改 0x2A1.teach_status 记录状态标志，
#     不会使机械臂进入零力拖动（进入固件原生零力拖动请用 6.1 的 set_leader_mode）。
from pyAgxArm.protocols.can_protocol.msgs.piper.default import ArmMsgMotionCtrl

robot._send_msg(ArmMsgMotionCtrl(grag_teach_ctrl=1))  # 1=开始记录, 2=结束记录
time.sleep(0.1)
print("grag_teach_ctrl=1 -> teach_status:", robot.get_arm_status().msg.teach_status)

robot._send_msg(ArmMsgMotionCtrl(grag_teach_ctrl=2))
time.sleep(0.1)
print("grag_teach_ctrl=2 -> teach_status:", robot.get_arm_status().msg.teach_status)

---
## 7. 设备参数查询、配置与关节标定 (`0x472`～`0x488`)

In [ ]:
# 7.1 查询关节角度/速度上限 (0x472->0x473) 与最大加速度上限 (0x472->0x47C)
# 注：参数查询函数默认有 min_interval=1.0s 请求节流，连续查询需显式传 min_interval=0.0。
for i in range(1, 7):
    av = robot.get_joint_angle_vel_limits(i, min_interval=0.0)
    ac = robot.get_joint_acc_limits(i, min_interval=0.0)
    print(f"J{i}:", av.msg if av else None, "|", ac.msg if ac else None)

In [ ]:
# 7.2 写入关节角度/速度上限与加速度上限 (0x474 / 0x475)
# 注：内部发送设置帧后会等待 0x476 ACK 并自动重查回读比对，确认生效才返回 True；不修改的项可传 None（底层填 0x7FFF）。
av6 = robot.get_joint_angle_vel_limits(6, min_interval=0.0)
ac6 = robot.get_joint_acc_limits(6, min_interval=0.0)
if av6 is not None:
    print("set_joint_angle_vel_limits(J6):", robot.set_joint_angle_vel_limits(
        6, av6.msg.min_angle_limit, av6.msg.max_angle_limit, av6.msg.max_joint_spd))
if ac6 is not None:
    print("set_joint_acc_limits(J6)      :", robot.set_joint_acc_limits(6, ac6.msg.max_joint_acc))
# robot.set_joint_angle_vel_acc_limits_to_default()

In [ ]:
# 7.3 查询与写入末端线速度/角速度/加速度上限 (0x477.B0=1 -> 0x478 / 0x479)
# 注：这是设备配置的物理最大速度/加速度上限，不同于 3.1 中 0x151.B2 的 speed_percent 百分比。
fl = robot.get_flange_vel_acc_limits(min_interval=0.0)
print("末端速度/加速度限制:", fl.msg if fl else None)
if fl is not None:
    print("set_flange_vel_acc_limits:", robot.set_flange_vel_acc_limits(
        fl.msg.end_max_linear_vel, fl.msg.end_max_angular_vel,
        fl.msg.end_max_linear_acc, fl.msg.end_max_angular_acc))
# robot.set_flange_vel_acc_limits_to_default()

In [ ]:
# 7.4 查询与写入碰撞防护等级 (0x47A/0x47B, 0..8) 与关节助力等级 (0x487/0x488, 0..10)
# 注：等级 0 表示关闭碰撞检测 / 不助力；写入单轴时 SDK 会先读取当前 6 轴等级，修改指定轴后整帧写回并回读校验。
crash = robot.get_crash_protection_rating(min_interval=0.0)
assist = robot.get_joint_assistance_rating(min_interval=0.0)
print("碰撞防护等级 (J1..J6):", crash.msg if crash else None)
print("关节助力等级 (J1..J6):", assist.msg if assist else None)
if crash is not None:
    print("set_crash_protection_rating(J1):", robot.set_crash_protection_rating(1, crash.msg[0]))
if assist is not None:
    print("set_joint_assistance_rating(J1):", robot.set_joint_assistance_rating(1, assist.msg[0]))

In [ ]:
# 7.5 设置固件负载档位 (set_payload: 'empty' | 'half' | 'full') 与 0x48x 周期反馈开关
# 注：两者均为 ACK-only 接口（收到 0x476 的 0x77 指令确认即返回 True）；0x481~0x486 反馈全为 0 无意义，应保持 False 关闭。
print("set_payload:", robot.set_payload(robot.OPTIONS.PAYLOAD.EMPTY))
print("set_links_vel_acc_period_feedback:", robot.set_links_vel_acc_period_feedback(False))

In [ ]:
# 7.6 清除关节错误 (clear_joint_error) 与关节零点标定 (calibrate_joint, 0x475)
# ⚠️ 警告：calibrate_joint 会把关节当前物理位置直接覆写为硬件零点！切勿随意执行：
print("clear_joint_error:", robot.clear_joint_error(255))
# robot.calibrate_joint(6)

---
## 8. 断开连接

In [ ]:
# 注：disconnect() 仅停止主机接收线程并关闭 CAN 句柄，不会自动将机械臂失能或改变固件当前模式；
#     若此时处于 Follower 受控状态会继续维持位置锁死，如需彻底断电失能请先取消下面两行注释（注意扶稳机械臂防下坠）：
# gripper.disable_gripper()
# robot.disable(255)
robot.disconnect()
print("Connected:", robot.is_connected())